# Phase 3, step 2: v1, one tracking policy on the whole training set

The smoke test showed one policy can track 10 motions with the course's task and settings unchanged.
v1 is the same recipe on **every training motion in dataset v1** (~978 motions with their mirrors
and time-scaled copies, ~78 min of motion), still unchanged: 4096 envs, the same network, rewards
and randomization.

- **Training goes in chunks** (default 5,000 iterations per run of the train cell, up to a target,
  default 20,000). Checkpoints are saved every 1,000 iterations to Drive and W&B. Run the train cell
  again, in this session or a new one, and it carries on from the newest checkpoint, so a Colab
  disconnect costs at most 1,000 iterations.
- **Scoring:** after a chunk, the newest checkpoint is scored on a **fixed evaluation set**: every
  validation original (never trained on, not even mirrored) plus 30 training originals. Each is played
  with `play_motion`, the course's scorer.
- **Done (Phase 3)** when v1 tracks its training motions about as well as a policy trained on only a
  few. Our stand-in for that is the smoke test: median ~1 cm and ~3°, 10/10 HOLDS UP.

Needs a **G4 runtime**. **Nothing here moves the robot;** a new policy needs a TA's OK before the robot.

In [ ]:
# @title Setup: pupper-mjlab (pinned), kflib, Drive (about 2 minutes)
import os, sys, subprocess
COMMIT = "aa475e9a393a209b29fd5de7e0ee10f3de643230"
BRANCH = "final-project-kflib"  # @param {type:"string"}
DATASET_NAME = "v1"  # @param {type:"string"}
RUN_NAME = "pgt_v1"  # @param {type:"string"}
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: switch to a G4 runtime")

if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/cs123-stanford/pupper-mjlab /content/pupper-mjlab
!git -C /content/pupper-mjlab checkout -q {COMMIT}
!pip install -q -e /content/pupper-mjlab
sys.path.insert(0, "/content/pupper-mjlab/src")
if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} https://github.com/MlsBli/gemini_pupper /content/gemini_pupper
!git -C /content/gemini_pupper pull -q
sys.path.insert(0, "/content/gemini_pupper/final_project")

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pupper_general_tracker")
DATASET = ROOT / "datasets" / DATASET_NAME
RUNS = ROOT / "runs"
EVAL_DIR = RUNS / f"eval_{RUN_NAME}"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
from kflib.dataset import Dataset
ds = Dataset(DATASET)
print(len(ds.rows()), "motions in", DATASET)

In [ ]:
# @title Log into W&B (key from Colab Secrets: WANDB_API_KEY)
import wandb
try:
  from google.colab import userdata
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:  # noqa: BLE001
  pass
wandb.login()
ENTITY = wandb.Api().default_entity
print("W&B entity:", ENTITY)

In [ ]:
# @title The training library (built and uploaded once; about 5 minutes the first time)
import json
from kflib.dataset import build_library
from kflib.runs import library_ids
LIB_NAME = f"{RUN_NAME}_train"
LIB_DIR = ROOT / "libraries"
LIB_DIR.mkdir(parents=True, exist_ok=True)
LIB, LIB_JSON = LIB_DIR / f"{LIB_NAME}.npz", LIB_DIR / f"{LIB_NAME}.json"
ids = library_ids(ds)
if LIB.exists():
  assert json.loads(LIB_JSON.read_text())["ids"] == ids, "the dataset changed since this library was built: use a new RUN_NAME"
  print("library exists:", LIB)
else:
  library = build_library({i: ds.load_motion(i) for i in ids}, LIB)
  LIB_JSON.write_text(json.dumps(dict(dataset=str(DATASET), ids=ids, clips=library.clips)))
  print(f"built: {len(ids)} motions, {len(library.root_pos) / 50 / 60:.0f} min ->", LIB)

MOTION_ARTIFACT = f"{ENTITY}/mjlab/{LIB_NAME}"
try:
  wandb.Api().artifact(MOTION_ARTIFACT + ":latest")
  print("already on W&B:", MOTION_ARTIFACT)
except Exception:  # noqa: BLE001  (not uploaded yet)
  !python -m mjlab.tasks.tracking.config.pupper.upload_motion {LIB} --name {LIB_NAME} 2>/dev/null | tail -2

In [ ]:
# @title The fixed evaluation set, and its plain-replay scores (once; ~15 minutes the first time)
from kflib.runs import eval_set
from kflib.score import score
EVAL_JSON = EVAL_DIR / "eval_set.json"
if EVAL_JSON.exists():
  EVAL = json.loads(EVAL_JSON.read_text())
else:
  EVAL = [dict(r, npz=str(ds.motions / f"{r['id']}.npz")) for r in eval_set(ds)]
  EVAL_JSON.write_text(json.dumps(EVAL, indent=1))
from collections import Counter
print(len(EVAL), "motions:", dict(Counter((r["split"], r["source"]) for r in EVAL)))
WORKERS = min(8, os.cpu_count())
PLAIN = EVAL_DIR / "plain.csv"
if not PLAIN.exists():
  score(EVAL, None, out_csv=PLAIN, workers=WORKERS)
print("plain replay scores:", PLAIN)

## Train, in chunks

Each run of this cell trains up to `CHUNK` more iterations (never past `TARGET`), continuing from the
newest checkpoint of `RUN_NAME`. A 5,000-iteration chunk should take roughly 25-35 min if v1 runs at the
smoke test's speed (3,000 in ~15 min): check the iterations per second in the log and adjust.
Watch `Metrics/motion/error_*` and `Episode_Reward/motion_*` on W&B. Every checkpoint also writes a
`policy.json` with the whole library inside (~60 MB): that's the course's trick runner, harmless, and
not the file to deploy.

In [ ]:
# @title Train (continues from the newest checkpoint)
TARGET = 20000  # @param {type:"integer"}
CHUNK = 5000  # @param {type:"integer"}
import re
from kflib.runs import chunk_plan, latest_checkpoint
last, ckpt = latest_checkpoint(RUNS, RUN_NAME)  # model_4999.pt = 5,000 iterations done (rsl_rl counts from 0)
todo, end = chunk_plan(last, ckpt is not None, TARGET, CHUNK)
print(f"{RUN_NAME}: " + (f"{last + 1} iterations done ({ckpt.parent.name}/{ckpt.name})" if ckpt else "starting")
      + (f", training up to {end + 1}" if todo else ": TARGET reached (raise it to keep going)"))
if todo > 0:
  os.chdir("/content/pupper-mjlab")
  from mjlab.scripts.train import TrainConfig, launch_training
  from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
  import mjlab.tasks  # registers the tasks
  TASK = "Mjlab-Tracking-Flat-Pupper-v3"
  env_cfg = load_env_cfg(TASK)
  env_cfg.scene.num_envs = 4096
  agent_cfg = load_rl_cfg(TASK)
  agent_cfg.max_iterations = todo  # learn() runs this many MORE iterations after a resume
  agent_cfg.save_interval = 1000
  agent_cfg.logger = "wandb"
  agent_cfg.run_name = RUN_NAME
  if ckpt:  # resume from exactly the newest checkpoint (any session's folder)
    agent_cfg.resume = True
    agent_cfg.load_run = re.escape(ckpt.parent.name)
    agent_cfg.load_checkpoint = re.escape(ckpt.name)
  launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg, registry_name=MOTION_ARTIFACT, log_root=str(RUNS)))
  print("now:", latest_checkpoint(RUNS, RUN_NAME)[0] + 1, "iterations done")

## Score the newest checkpoint

Plays all evaluation motions with the policy (~15 min), next to plain replay. Scores are kept per
checkpoint (`eval_<iteration>.csv`), and the last cell compares them all, so you can see whether more
training still helps.

In [ ]:
# @title Score the newest checkpoint on the evaluation set
import csv
from kflib.runs import latest_checkpoint, summarize
from kflib.score import score
it, ckpt = latest_checkpoint(RUNS, RUN_NAME)
out = EVAL_DIR / f"eval_{it}.csv"
if out.exists():
  print("already scored:", out)
else:
  score(EVAL, ckpt, out_csv=out, workers=WORKERS, plain=False)
rows = list(csv.DictReader(out.open()))
for split, s in summarize(rows).items():
  print(f"\n{split}: {s['holds_up']}/{s['n']} HOLDS UP, {s['wobbles']} WOBBLES, {s['falls']} FALLS"
        f" | median {s['median_cm']} cm {s['median_deg']} deg | worst {s['worst_cm']} cm {s['worst_deg']} deg")
  for w in s["worst"]:
    print("   ", w)
plain = summarize([dict(r, mode="policy") for r in csv.DictReader(PLAIN.open())])
print("\nplain replay, for comparison:", {k: f"{v['holds_up']}/{v['n']} HOLDS UP, median {v['median_cm']} cm {v['median_deg']} deg" for k, v in plain.items()})

In [ ]:
# @title Progress across checkpoints
import csv, re
from kflib.runs import summarize
for f in sorted(EVAL_DIR.glob("eval_*.csv"), key=lambda p: int(re.findall(r"\d+", p.stem)[0])):
  s = summarize(list(csv.DictReader(f.open())))
  print(f"{f.stem:12}", " | ".join(f"{k}: {v['holds_up']}/{v['n']} HOLDS UP, median {v['median_cm']} cm {v['median_deg']} deg" for k, v in s.items()))

In [ ]:
# @title (Optional) Watch the policy on one motion (open the share link; stop the cell when done)
from kflib.runs import latest_checkpoint
WATCH = EVAL[0]["id"]  # @param {type:"string"}
it, ckpt = latest_checkpoint(RUNS, RUN_NAME)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {ds.motions / (WATCH + '.npz')} --checkpoint {ckpt} --viewer viser --share True --loops 2